In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

In [6]:
df = pd.read_csv("../../../../scripts/notebooks/Ankita/ML/loan_data.csv")
df.head(1)

,Customer_ID,Loan_Amount,Customer_Vintage,No_of_MFIs,Category,Region,Income,Credit_Score,CREATED_TS,Final_Status
0,1,131958,0,2,Unique,South,65563,629,2025-01-01 00:00:00,Approved


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Customer_ID       20000 non-null  int64 
 1   Loan_Amount       20000 non-null  int64 
 2   Customer_Vintage  20000 non-null  int64 
 3   No_of_MFIs        20000 non-null  int64 
 4   Category          20000 non-null  object
 5   Region            20000 non-null  object
 6   Income            20000 non-null  int64 
 7   Credit_Score      20000 non-null  int64 
 8   CREATED_TS        20000 non-null  object
 9   Final_Status      20000 non-null  object
dtypes: int64(6), object(4)
memory usage: 1.5+ MB
None


# Data Cleaning

In [9]:
# Convert date column
df['CREATED_TS'] = pd.to_datetime(df['CREATED_TS'], errors='coerce')

# Create Month column (important for your use case)
df['Month'] = df['CREATED_TS'].dt.to_period('M')

# Handle missing values
df = df.dropna(subset=['Loan_Amount', 'Final_Status'])

# Fill categorical nulls
df['Category'] = df['Category'].fillna('Unknown')

# Feature Engineering

## Loan Amount Bucket

In [10]:
def loan_bucket(x):
    if x <= 50000:
        return '0-50K'
    elif x <= 100000:
        return '50K-100K'
    elif x <= 200000:
        return '100K-200K'
    else:
        return '200K+'

df['Loan_Bucket'] = df['Loan_Amount'].apply(loan_bucket)

## Customer Vintage Bucket

In [ ]:
def vintage_bucket(x):
    if x <= 2:
        return '0-2 years'
    elif x <= 4:
        return '2-4 years'
    elif x <= 6:
        return '4-6 years'
    else:
        return '>6 years'

df['Customer_Vintage_Bucket'] = df['Customer_Vintage'].apply(vintage_bucket)

# Exploratory Data Analysis (EDA)

In [ ]:
#Rejection Count by Month
rejection_trend = df[df['Final_Status'] == 'Rejected'].groupby('Month').size()

rejection_trend.plot(kind='line', figsize=(10,5))
plt.title("Month-wise Rejection Trend")
plt.show()

In [ ]:
#Rejection by Category
category_reject = pd.crosstab(df['Category'], df['Final_Status'])

category_reject.plot(kind='bar', stacked=True, figsize=(10,6))
plt.title("Rejection by Category")
plt.show()

In [ ]:
pivot = pd.pivot_table(
    df,
    values='Loan_Amount',
    index='Category',
    columns='Customer_Vintage_Bucket',
    aggfunc='sum',
    margins=True
)

print(pivot)

# Data Preparation for Model

In [11]:
df['Target'] = df['Final_Status'].apply(lambda x: 1 if x == 'Rejected' else 0)

In [ ]:
le = LabelEncoder()

df['Category_enc'] = le.fit_transform(df['Category'])
df['Loan_Bucket_enc'] = le.fit_transform(df['Loan_Bucket'])
df['Vintage_enc'] = le.fit_transform(df['Customer_Vintage_Bucket'])

In [ ]:
features = ['Loan_Amount', 'No_of_MFIs', 'Category_enc', 'Loan_Bucket_enc', 'Vintage_enc']
X = df[features]
y = df['Target']

In [ ]:
# Train, test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Model Building (Logistic Regression)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Predictions & Evaluation
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Feature Importance
importance = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print(importance)